# IM3 Tethys Outputs Analysis

This notebook analyzes the outputs from IM3 Tethys runs. It includes functions to read output files, summarize data, and plot results.

In [17]:
import xarray as xr
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors
import numpy as np

# scenarios = [
#     'rcp45cooler_ssp3', 'rcp45cooler_ssp5',
#     'rcp45hotter_ssp3', 'rcp45hotter_ssp5',
#     'rcp85cooler_ssp3', 'rcp85cooler_ssp5',
#     'rcp85hotter_ssp3', 'rcp85hotter_ssp5',
# ]

# sum for a given sector, type and year
import xarray as xr

import xarray as xr

def sector_sum(outputs_dir, scenario, sector, type, year):
    """
    Compute summed values for a given sector, type, and year using sector-specific logic.
    Handles datasets with or without monthly dimensions.
    """
    config = {
        "Electricity": lambda ds: [v for v in ds.data_vars if v.startswith("electricity_")],
        "Irrigation": lambda ds: [v for v in ds.data_vars if v[0].isupper()],
        "Domestic": lambda ds: [v for v in ds.data_vars if v.startswith("Domestic")],
        "Livestock": lambda ds: [v for v in ds.data_vars if v[0].isupper()],
        "Manufacturing": lambda ds: [v for v in ds.data_vars if v.startswith("Manufacturing")],
        "Mining": lambda ds: [v for v in ds.data_vars if v.startswith("Mining")],
    }
    if sector not in config:
        raise ValueError(f"Unknown sector: {sector}")
    ds = xr.open_dataset(f"{outputs_dir}/{scenario}/{sector}_{type}.nc")
    selected_vars = config[sector](ds)
    if not selected_vars:
        raise ValueError(f"No valid variables for sector '{sector}' and type '{type}'.")
    
    # Sum across 'month' dimension if present, then collapse over lat/lon
    if "month" in ds.dims:
        return sum(ds[v] for v in selected_vars).sum(dim=["lat", "lon", "month"]).sel(year=year)
    else:
        # Sum directly across lat/lon for yearly datasets
        return sum(ds[v] for v in selected_vars).sum(dim=["lat", "lon"]).sel(year=year)

# summarize all .nc files/sectors in the scenario folder
def summarize_sectors(outputs_dir, scenario, year):
    """
    Summarize all .nc files in the specified scenario folder into a table.
    Handles both yearly and monthly outputs, collapsing monthly data into yearly results.
    Filters unsupported files like 'gridded_runoff_shares.nc'.
    """
    results = []
    for file_name in os.listdir(f"{outputs_dir}/{scenario}"):
        if file_name.endswith(".nc"):  # Process only NetCDF files
            if "gridded_runoff_shares" in file_name:
                continue  # Skip unsupported files
            
            try:
                # Parse sector and type from the file name
                parts = file_name[:-3].split("_")  # Remove ".nc" file extension and split
                if len(parts) >= 2:
                    sector, type = parts[0], parts[1]
                    
                    # Call the sector_sum function
                    sector_sum_result = sector_sum(outputs_dir, scenario, sector, type, year)
                    
                    if sector_sum_result is not None:
                        results.append({
                            "File": file_name,
                            "Sector": sector,
                            "Type": type,
                            "Year": year,
                            "Sum": float(sector_sum_result.values) if sector_sum_result.size == 1 else None,
                        })
            except Exception as e:
                # Log error and continue to next file
                results.append({
                    "File": file_name,
                    "Sector": None,
                    "Type": None,
                    "Year": year,
                    "Sum": None,
                    "Error": str(e),
                })
    return pd.DataFrame(results)

In [18]:
# params
outputs_dir = '../../output_test'
scenario = 'rcp45cooler_ssp3'
sector = 'Electricity'
type = 'withdrawals'
year = 2020


In [19]:
# take a look at one output
df = xr.open_dataset(f'{outputs_dir}/{scenario}/{sector}_{type}.nc') 
df


<xarray.Dataset> Size: 41MB
Dimensions:                      (year: 7, lat: 224, lon: 464)
Coordinates:
  * lat                          (lat) float64 2kB 52.94 52.81 ... 25.19 25.06
  * lon                          (lon) float64 4kB -124.9 -124.8 ... -67.06
  * year                         (year) int64 56B 2020 2025 2030 ... 2045 2050
    spatial_ref                  int64 8B ...
Data variables:
    electricity_biomass          (year, lat, lon) float64 6MB ...
    electricity_coal             (year, lat, lon) float64 6MB ...
    electricity_gas              (year, lat, lon) float64 6MB ...
    electricity_geothermal       (year, lat, lon) float64 6MB ...
    electricity_nuclear          (year, lat, lon) float64 6MB ...
    electricity_refined liquids  (year, lat, lon) float64 6MB ...
    electricity_solar            (year, lat, lon) float64 6MB ...

In [20]:
electricity_sum = sector_sum(outputs_dir, scenario, sector="Electricity", type=type, year=year)
electricity_sum

<xarray.DataArray ()> Size: 8B
array(115.62228302)
Coordinates:
    year         int64 8B 2020
    spatial_ref  int64 8B 0

2020 comparisons 

In [21]:
summary_table = summarize_sectors(outputs_dir, scenario, year).sort_values(by=["Sector", "Type"])
summary_table

,File,Sector,Type,Year,Sum
12,Domestic_consumption.nc,Domestic,consumption,2020,9.336301
13,Domestic_consumption_monthly.nc,Domestic,consumption,2020,9.336301
0,Domestic_withdrawals.nc,Domestic,withdrawals,2020,59.153167
1,Domestic_withdrawals_monthly.nc,Domestic,withdrawals,2020,59.153167
14,Electricity_consumption.nc,Electricity,consumption,2020,4.403557
15,Electricity_consumption_monthly.nc,Electricity,consumption,2020,4.403557
2,Electricity_withdrawals.nc,Electricity,withdrawals,2020,115.622283
3,Electricity_withdrawals_monthly.nc,Electricity,withdrawals,2020,115.622283
16,Irrigation_consumption.nc,Irrigation,consumption,2020,79.391847
17,Irrigation_consumption_monthly.nc,Irrigation,consumption,2020,79.391847


In [22]:
summary_table = summarize_sectors(outputs_dir = '../../output_supersector', scenario=scenario, year=2020).sort_values(by=["Sector", "Type"])
summary_table

,File,Sector,Type,Year,Sum
12,Domestic_consumption.nc,Domestic,consumption,2020,9.336301
13,Domestic_consumption_monthly.nc,Domestic,consumption,2020,9.336301
0,Domestic_withdrawals.nc,Domestic,withdrawals,2020,59.153167
1,Domestic_withdrawals_monthly.nc,Domestic,withdrawals,2020,59.153167
14,Electricity_consumption.nc,Electricity,consumption,2020,4.403557
15,Electricity_consumption_monthly.nc,Electricity,consumption,2020,4.403557
2,Electricity_withdrawals.nc,Electricity,withdrawals,2020,115.622283
3,Electricity_withdrawals_monthly.nc,Electricity,withdrawals,2020,115.622283
16,Irrigation_consumption.nc,Irrigation,consumption,2020,79.379992
17,Irrigation_consumption_monthly.nc,Irrigation,consumption,2020,79.379992


In [25]:
summarize_sectors(outputs_dir = '../../output_sunflowerfix', scenario=scenario, year=2020).sort_values(by=["Sector", "Type"])

,File,Sector,Type,Year,Sum
12,Domestic_consumption.nc,Domestic,consumption,2020,9.336301
13,Domestic_consumption_monthly.nc,Domestic,consumption,2020,9.336301
0,Domestic_withdrawals.nc,Domestic,withdrawals,2020,59.153167
1,Domestic_withdrawals_monthly.nc,Domestic,withdrawals,2020,59.153167
14,Electricity_consumption.nc,Electricity,consumption,2020,4.403557
15,Electricity_consumption_monthly.nc,Electricity,consumption,2020,4.403557
2,Electricity_withdrawals.nc,Electricity,withdrawals,2020,115.622283
3,Electricity_withdrawals_monthly.nc,Electricity,withdrawals,2020,115.622283
16,Irrigation_consumption.nc,Irrigation,consumption,2020,79.379992
17,Irrigation_consumption_monthly.nc,Irrigation,consumption,2020,79.379992


2050 comparison 

In [23]:
summary_table = summarize_sectors(outputs_dir = '../../output_test', scenario=scenario, year=2050).sort_values(by=["Sector", "Type"])
summary_table


,File,Sector,Type,Year,Sum
12,Domestic_consumption.nc,Domestic,consumption,2050,9.173410
13,Domestic_consumption_monthly.nc,Domestic,consumption,2050,9.173410
0,Domestic_withdrawals.nc,Domestic,withdrawals,2050,56.833293
1,Domestic_withdrawals_monthly.nc,Domestic,withdrawals,2050,56.833293
14,Electricity_consumption.nc,Electricity,consumption,2050,1.245051
15,Electricity_consumption_monthly.nc,Electricity,consumption,2050,1.245051
2,Electricity_withdrawals.nc,Electricity,withdrawals,2050,6.602648
3,Electricity_withdrawals_monthly.nc,Electricity,withdrawals,2050,6.602648
16,Irrigation_consumption.nc,Irrigation,consumption,2050,96.133443
17,Irrigation_consumption_monthly.nc,Irrigation,consumption,2050,96.133443


In [24]:
summary_table = summarize_sectors(outputs_dir = '../../output_supersector', scenario=scenario, year=2050).sort_values(by=["Sector", "Type"])
summary_table

,File,Sector,Type,Year,Sum
12,Domestic_consumption.nc,Domestic,consumption,2050,9.173410
13,Domestic_consumption_monthly.nc,Domestic,consumption,2050,9.173410
0,Domestic_withdrawals.nc,Domestic,withdrawals,2050,56.833293
1,Domestic_withdrawals_monthly.nc,Domestic,withdrawals,2050,56.833293
14,Electricity_consumption.nc,Electricity,consumption,2050,1.245051
15,Electricity_consumption_monthly.nc,Electricity,consumption,2050,1.245051
2,Electricity_withdrawals.nc,Electricity,withdrawals,2050,6.602648
3,Electricity_withdrawals_monthly.nc,Electricity,withdrawals,2050,6.602648
16,Irrigation_consumption.nc,Irrigation,consumption,2050,96.133443
17,Irrigation_consumption_monthly.nc,Irrigation,consumption,2050,96.133443


In [27]:
summarize_sectors(outputs_dir = '../../output_sunflowerfix', scenario=scenario, year=2050).sort_values(by=["Sector", "Type"])

,File,Sector,Type,Year,Sum
12,Domestic_consumption.nc,Domestic,consumption,2050,9.173410
13,Domestic_consumption_monthly.nc,Domestic,consumption,2050,9.173410
0,Domestic_withdrawals.nc,Domestic,withdrawals,2050,56.833293
1,Domestic_withdrawals_monthly.nc,Domestic,withdrawals,2050,56.833293
14,Electricity_consumption.nc,Electricity,consumption,2050,1.245051
15,Electricity_consumption_monthly.nc,Electricity,consumption,2050,1.245051
2,Electricity_withdrawals.nc,Electricity,withdrawals,2050,6.602648
3,Electricity_withdrawals_monthly.nc,Electricity,withdrawals,2050,6.602648
16,Irrigation_consumption.nc,Irrigation,consumption,2050,96.133443
17,Irrigation_consumption_monthly.nc,Irrigation,consumption,2050,96.133443
